# Import des librairies

In [81]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

import optuna

import matplotlib.pyplot as plt

import mlflow
import dagshub

# Chargement du dataset

In [82]:
url = r"https://drive.google.com/uc?export=download&id=1YkRYSkAHeH7nMgO-ums8zbU7paODGs3t"
df = pd.read_csv(url)

# Feature engineering

In [83]:
# Modification du type de la Serie "admission_date" : object => datetime
df["admission_date"] = pd.to_datetime(df["admission_date"])

# Création d'une Serie "discharge_date" (= admission_date + length_of_stay)
df["date_sortie"] = df["admission_date"] + pd.to_timedelta(df["length_of_stay"], unit="D")

# Séparation de la viariable cible et des variables explicatives

In [84]:
features_list = [
    "season", 
    "age", 
    "gender", 
    "region", 
    "primary_diagnosis", 
    "comorbidities_count", 
    "length_of_stay", 
    "treatment_type", 
    "medications_count", 
    "followup_visits_last_year",
    "prev_readmissions", 
    "insurance_type",
    "discharge_disposition"
    ]
target_variable = "label"

X = df.loc[:,features_list]
Y = df.loc[:,target_variable]

# Preprocessing

In [85]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns

In [86]:
# Pipeline - variables numériques
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline - variables catégorielles
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Pipeline - préprocesseur à appliquer au variables explicatives
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_features),
    ("cat", cat_pipeline, categorical_features)
])

In [87]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 2, 10)

    model_study = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf
        ))
    ])

    score = cross_val_score(model_study, X_train, Y_train, cv=3, scoring = "recall").mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

best_params = study.best_params

[I 2026-03-19 17:06:04,241] A new study created in memory with name: no-name-84ea55e7-c2d7-444e-add4-cb8b6b25cf28
[I 2026-03-19 17:06:04,778] Trial 0 finished with value: 0.9675925925925926 and parameters: {'n_estimators': 53, 'max_depth': 8, 'min_samples_split': 9, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.9675925925925926.
[I 2026-03-19 17:06:05,854] Trial 1 finished with value: 0.9810789049919485 and parameters: {'n_estimators': 160, 'max_depth': 4, 'min_samples_split': 9, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.9810789049919485.
[I 2026-03-19 17:06:06,752] Trial 2 finished with value: 0.947866344605475 and parameters: {'n_estimators': 74, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.9810789049919485.
[I 2026-03-19 17:06:08,110] Trial 3 finished with value: 0.9774557165861513 and parameters: {'n_estimators': 167, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.98

In [88]:
best_model = RandomForestClassifier(**best_params)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

In [89]:
dagshub.init(repo_owner='quentin.seng67', repo_name="Hospital_Readmission_Project", mlflow=True)
mlflow.set_experiment("readmission_prediction")

Initialized MLflow to track repo "quentin.seng67/Hospital_Readmission_Project"

Repository quentin.seng67/Hospital_Readmission_Project initialized!

<Experiment: artifact_location='mlflow-artifacts:/937f685cc5e0417596c44c61948e6f35', creation_time=1773926632209, experiment_id='0', last_update_time=1773926632209, lifecycle_stage='active', name='readmission_prediction', tags={}, workspace='default'>

In [90]:
with mlflow.start_run(run_name="random_forest_otpunaV2_recall"):

    model.fit(X_train, Y_train)

    Y_train_pred = model.predict(X_train)
    Y_test_pred = model.predict(X_test)

    # Prédictions
    Y_train_pred = model.predict(X_train)
    Y_test_pred = model.predict(X_test)

    # Probabilités
    Y_train_proba = model.predict_proba(X_train)[:, 1]
    Y_test_proba = model.predict_proba(X_test)[:, 1]

    # Métriques
    mlflow.log_metric("train_precision", precision_score(Y_train, Y_train_pred))
    mlflow.log_metric("test_precision", precision_score(Y_test, Y_test_pred))

    mlflow.log_metric("train_recall", recall_score(Y_train, Y_train_pred))
    mlflow.log_metric("test_recall", recall_score(Y_test, Y_test_pred))

    mlflow.log_metric("train_f1", f1_score(Y_train, Y_train_pred))
    mlflow.log_metric("test_f1", f1_score(Y_test, Y_test_pred))

    mlflow.log_metric("train_accuracy", accuracy_score(Y_train, Y_train_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(Y_test, Y_test_pred))
    
    # 👇 hyperparams
    mlflow.log_params(best_params)

    # confusion matrix
    cm_train = confusion_matrix(Y_train, Y_train_pred)
    cm_test = confusion_matrix(Y_test, Y_test_pred)

    # ----- TRAIN -----
    fig_train, ax_train = plt.subplots()
    ConfusionMatrixDisplay(cm_train).plot(ax=ax_train)
    ax_train.set_title("Confusion Matrix - Train")

    mlflow.log_figure(fig_train, "confusion_matrix_train.png")
    plt.close(fig_train)

    # ----- TEST -----
    fig_test, ax_test = plt.subplots()
    ConfusionMatrixDisplay(cm_test).plot(ax=ax_test)
    ax_test.set_title("Confusion Matrix - Test")

    mlflow.log_figure(fig_test, "confusion_matrix_test.png")
    plt.close(fig_test)


    # ROC train
    fpr_train, tpr_train, _ = roc_curve(Y_train, Y_train_proba)
    auc_train = auc(fpr_train, tpr_train)

    # ROC test
    fpr_test, tpr_test, _ = roc_curve(Y_test, Y_test_proba)
    auc_test = auc(fpr_test, tpr_test)

    # Plot
    fig, ax = plt.subplots()

    ax.plot(fpr_train, tpr_train, label=f"Train AUC = {auc_train:.3f}")
    ax.plot(fpr_test, tpr_test, label=f"Test AUC = {auc_test:.3f}")

    ax.plot([0, 1], [0, 1], linestyle="--")  # ligne aléatoire
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    ax.legend()

    # Log dans MLflow
    mlflow.log_figure(fig, "roc_curve.png")
    plt.close(fig)

    # Log des AUC
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_test", auc_test)


    print("\n✓ Run loggé dans mlflow")


✓ Run loggé dans mlflow
🏃 View run random_forest_otpunaV2_recall at: https://dagshub.com/quentin.seng67/Hospital_Readmission_Project.mlflow/#/experiments/0/runs/63b5b92d47a04ab4895f3b8f8a21626f
🧪 View experiment at: https://dagshub.com/quentin.seng67/Hospital_Readmission_Project.mlflow/#/experiments/0


In [91]:
print(classification_report(Y_test, Y_test_pred))

              precision    recall  f1-score   support

           0       0.83      0.08      0.14       385
           1       0.77      1.00      0.87      1215

    accuracy                           0.77      1600
   macro avg       0.80      0.54      0.50      1600
weighted avg       0.79      0.77      0.69      1600

